# WO-07 — WO-06 adjustments, class words, and the nearest river
Part A (so far): the ring sentences state only extremes, "all" and "all but one"; S6.4 is gone; S6.5 is widened; population density and human footprint are left out of the sentences (kept in the fill notes). Wording is in `profile_templates_v6_target.md` (the build target until the swap); the rules are in `scripts/edop/whg/wo06_rings.py`. Runs on the ring table saved in WO-06 (`output/edop/whg/wo06_ring_values.csv`); nothing is re-fetched. Read-only except `wo07_filled_profiles.md` and `output/edop/whg/wo07_fill.csv`.

In [ ]:
%matplotlib inline
# Cell 1 — Setup: wording file (the v6 target), server check, saved ring, flow and river tables
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C
from scripts.edop.whg import wo04_fill as W
from scripts.edop.whg import wo06_rings as R
from scripts.edop.whg import wo07_rivers as RV

W.TEMPLATES_PATH = ROOT / 'docs' / 'design' / 'dashboard' / 'profile_templates_v6_target.md'   # the build target until the swap
conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
FLOW = pd.read_csv(C.OUT / 'q2_flow_pairs.csv', dtype={'tag': str})
SAMPLE = C.population_points()
RING = R.load_table()
RIV = RV.load_table() if RV.RIVERS_CSV.exists() else None     # filled by Cell 5; Cell 2 uses it once it exists
print('commit', C.commit(), '| server:', W.get_json('/api/health'), '| wording:', W.templates_stamp(), '| ring table:', len(RING), 'basins | river table:', len(RIV) if RIV else 'not built yet', '| test places:', len(C.PLACES), '| sample points:', len(SAMPLE))

In [ ]:
# Cell 2 — The ten test places: the filled_profiles (written to docs/design/dashboard/wo07_filled_profiles.md) and their ring and river sentences
OUTMD = ROOT / 'docs' / 'design' / 'dashboard' / 'wo07_filled_profiles.md'
t0 = time.perf_counter()
TEST = {}
parts = ['# WO-07: the dummy filled from real data, ten test places (filled_profiles)', '',
         'Generated by `notebooks/edop/whg/wo07_adjust_classes_rivers.ipynb` (commit ' + C.commit() + '). Every `yes` unit of `modal_dummy_annotated.md` is filled from data; each `maybe` or `no` unit is a placeholder in the form `⟦id maybe: needs ...⟧`. Ring comparison units (S2.3, S5.2, S6.2, S6.3, S6.5) and the much-larger-river unit (S3.3) are silent where they have nothing true to say. Wording is the template file\'s, not polished.', '',
         'Wording from `' + W.templates_stamp() + '`.', '']
L = []
for name, (lat, lon) in C.PLACES.items():
    F = W.build_facts(lat, lon, conn, PAIRS, FLOW, ring=RING, rivers=RIV)
    if F['pub'] is None:
        L.append(f'{name}: no basin'); continue
    U = W.fill_units(F)
    TEST[name] = (F, U)
    parts.append(W.render_filled_profile(name, F, U)); parts.append('---'); parts.append('')
    L.append(f'{name} ({F["n_nb"]} neighbours; {U.get("S6.ring", {}).get("note", "")})')
    for u in ('S2.3', 'S5.2', 'S6.2', 'S6.3', 'S6.5', 'S3.1', 'S3.3'):
        if u in U:
            L.append(f'   {u}: ' + (U[u]['text'] if U[u]['ok'] else f'[silent: {U[u]["why"]}]'))
OUTMD.write_text('\n'.join(parts))
print(f'wrote {OUTMD.name} | {W.templates_stamp()} | {time.perf_counter() - t0:.0f}s\n\n' + '\n'.join(L))

In [4]:
# Cell 3 — The sample: 306 basin interior points through the ring rules; saved to a CSV (about 8 min)
rows, t_start = [], time.perf_counter()
for k, s in enumerate(SAMPLE, 1):
    F = W.build_facts(float(s['lat']), float(s['lon']), conn, PAIRS, FLOW, ring=RING)
    U = W.fill_units(F)
    d = dict(core_id=int(s['hybas_id_l6']), zone=s['climate_zone'], n_nb=F['n_nb'])
    for u in ('S2.3', 'S5.2', 'S6.2', 'S6.3', 'S6.5'):
        d[u] = int(U[u]['ok']); d[u + '_na'] = int(U[u].get('na', False)); d[u + '_why'] = U[u]['why'] or ''
    tr = {u: U[u].get('trace', {}) for u in ('S2.3', 'S5.2', 'S6.3', 'S6.5', 'S6.ring')}
    for u in ('S2.3', 'S5.2'):
        d[u + '_kind'], d[u + '_ext'], d[u + '_used'] = tr[u].get('kind', ''), len(tr[u].get('ext', [])), len(tr[u].get('used', []))
    d['S6.3_across'], d['S6.3_all'], d['S6.3_abo'] = tr['S6.3'].get('across', 0), len(tr['S6.3'].get('hit', [])), len(tr['S6.3'].get('abo', []))
    d['S6.3_first'] = (tr['S6.3'].get('hit') or tr['S6.3'].get('abo') or [('', '')])[0][0]
    d['S6.5_ext'], d['S6.5_used'] = len(tr['S6.5'].get('ext', [])), len(tr['S6.5'].get('used', []))
    d['S6.5_first'] = ' '.join(f'{a}' for a, b in tr['S6.5'].get('used', []))
    d['core_smallest'], d['core_largest'] = tr['S6.ring'].get('smallest'), tr['S6.ring'].get('largest')
    rows.append(d)
    if k % 50 == 0:
        print(f'  {k} of {len(SAMPLE)} in {time.perf_counter() - t_start:.0f}s')
S5 = pd.DataFrame(rows)
S5.to_csv(C.OUT / 'wo07_fill.csv', index=False)
print(f'rows saved: {len(S5)} | this run took {time.perf_counter() - t_start:.0f}s')

rows saved: 306 | this run took 471s


In [5]:
# Cell 4 — Firing rates over the sample: how often each unit speaks, what kind of sentence, how often the cap bites
L = [f'sample basins: {len(S5)}']
for u in ('S2.3', 'S5.2', 'S6.2', 'S6.3', 'S6.5'):
    L.append(f'{u}: speaks {int(S5[u].sum())} | silent (nothing true to say) {int(S5[u + "_na"].sum())} | could not fill {int(((S5[u] == 0) & (S5[u + "_na"] == 0)).sum())}')
L.append('')
for u, cap in (('S2.3', R.CAP['S2_3']), ('S5.2', R.CAP['S5_2'])):
    L.append(f'{u}: kind {S5[u + "_kind"].value_counts().to_dict()} | extremes found per basin {S5[u + "_ext"].value_counts().sort_index().to_dict()} | cap {cap} bites at {int((S5[u + "_ext"] > cap).sum())}')
L.append(f'S6.5: extremes found per basin {S5["S6.5_ext"].value_counts().sort_index().to_dict()} | cap {R.CAP["S6_5"]} bites at {int((S5["S6.5_ext"] > R.CAP["S6_5"]).sum())} of {int((S5["S6.5"] == 1).sum())} that speak')
spoke = S5[S5['S6.3'] == 1]
L.append(f'S6.3: speaks at {len(spoke)} ("all" {int((spoke["S6.3_all"] > 0).sum())}, "all but one" {int(((spoke["S6.3_all"] == 0) & (spoke["S6.3_abo"] > 0)).sum())}) | first measure: ' + str(spoke['S6.3_first'].value_counts().to_dict()))
sp = S5[S5['S6.5'] == 1]
L.append('S6.5 measure named first: ' + str(sp['S6.5_first'].str.split().str[0].value_counts().to_dict()))
L.append('')
S5['nb_band'] = pd.cut(S5['n_nb'], [-1, 2, 4, 6, 8, 99], labels=['0-2', '3-4', '5-6', '7-8', '9+'])
S5['any_sentence'] = ((S5['S2.3'] + S5['S5.2'] + S5['S6.3'] + S5['S6.5']) > 0).astype(int)
S5['n_sentences'] = S5['S2.3'] + S5['S5.2'] + S5['S6.3'] + S5['S6.5']
g = S5.groupby('nb_band', observed=True)[['S2.3', 'S5.2', 'S6.3', 'S6.5', 'any_sentence']].mean()
g.insert(0, 'basins', S5.groupby('nb_band', observed=True).size())
L.append('by number of neighbours (share of basins that speak):'); L.append(g.round(2).to_string())
L.append('ring sentences per basin (S2.3, S5.2, S6.3, S6.5; the flow sentence S6.2 not counted): ' + str(S5['n_sentences'].value_counts().sort_index().to_dict()))
print('\n'.join(L))

sample basins: 306
S2.3: speaks 262 | silent (nothing true to say) 44 | could not fill 0
S5.2: speaks 161 | silent (nothing true to say) 145 | could not fill 0
S6.2: speaks 298 | silent (nothing true to say) 8 | could not fill 0
S6.3: speaks 105 | silent (nothing true to say) 201 | could not fill 0
S6.5: speaks 159 | silent (nothing true to say) 147 | could not fill 0

S2.3: kind {'count': 153, 'extreme': 59, 'allbutone': 50, '': 28, 'count_unstated': 16} | extremes found per basin {0: 247, 1: 35, 2: 15, 3: 9} | cap 2 bites at 9
S5.2: kind {'none': 90, 'count': 76, 'extreme': 45, 'allbutone': 40, '': 28, 'count_unstated': 27} | extremes found per basin {0: 261, 1: 35, 2: 10} | cap 2 bites at 0
S6.5: extremes found per basin {0: 147, 1: 60, 2: 40, 3: 24, 4: 18, 5: 8, 6: 6, 7: 2, 8: 1} | cap 3 bites at 35 of 159 that speak
S6.3: speaks at 105 ("all" 70, "all but one" 35) | first measure: {'elev': 60, 'precip': 24, 'temp': 14, 'aridity': 7}
S6.5 measure named first: {'elev': 47, 'relief':

In [ ]:
# Cell 5 — Part C: the river table for the ten places, the 306 sample basins and the 258 World Heritage Cities (every cutoff tried is kept; resumable; a failing point is logged and skipped)
CITIES = pd.DataFrame(conn.execute("SELECT id, city, country, ST_Y(geom::geometry), ST_X(geom::geometry) FROM gaz.wh_cities ORDER BY id").fetchall(), columns=['id', 'city', 'country', 'lat', 'lon'])
points = ([(n, 'test', la, lo) for n, (la, lo) in C.PLACES.items()]
          + [(f"sample {int(s['hybas_id_l6'])}", 'sample', float(s['lat']), float(s['lon'])) for s in SAMPLE]
          + [(f"{r.city}, {r.country}", 'wh_cities', float(r.lat), float(r.lon)) for r in CITIES.itertuples()])
t0 = time.perf_counter()
n_rows, n_new, failed = RV.fetch_table(points, conn)
RIV = RV.load_table()
print(f'river table: {n_rows} points in {RV.RIVERS_CSV.name} | {n_new} fetched this run | {len(failed)} failed | {time.perf_counter() - t0:.0f}s')
for lab, err in failed[:20]:
    print('  FAILED:', lab, '|', err)

In [ ]:
# Cell 6 — Part C at every cutoff tried: distance to the nearest river, how often it is named, how often a much larger river is near
T = pd.DataFrame(list(RIV.values()))
nm_ok = lambda r, p, x: isinstance(r[f'{p}_name'], str) and r[f'{p}_name_km'] <= x and not (isinstance(r[f'{p}_name2'], str) and r[f'{p}_name2_km'] <= x)
L = []
for grp in ('sample', 'wh_cities'):
    d = T[(T['group'] == grp) & T['hid'].notna()]
    n = len(d)
    L.append(f'== {grp}: {n} points with a basin ==')
    for c in RV.CLASSES:
        km = d[f'r{c}_km']
        named = {x: int(sum(nm_ok(r, f'r{c}', x) for r in d.to_dict('records'))) for x in (3, 5, 10)}
        L.append(f'S3.1, nearest river of class {c} or larger: distance median {km.median():.1f} km | 90% {km.quantile(0.9):.1f} | within 1 km {int((km <= 1).sum())} | within 10 km {int((km <= 10).sum())} | beyond 50 km {int((km > 50).sum())}')
        L.append(f'     named (Natural Earth line within x km of the reach, no second name that close): ' + ' | '.join(f'{x} km {v} ({100 * v / n:.0f}%)' for x, v in named.items()))
        amb = int(sum(isinstance(r[f'r{c}_name'], str) and r[f'r{c}_name_km'] <= 5 and isinstance(r[f'r{c}_name2'], str) and r[f'r{c}_name2_km'] <= 5 for r in d.to_dict('records')))
        L.append(f'     ambiguous (two different names within 5 km, so the sentence says "a river"): {amb}')
    L.append(f'S3.3, a reach {RV.STEP} classes above the largest in the own basin ({RV.OWN_MIN_KM:g} km or more inside it): own-basin largest class {d["own_min"].value_counts().sort_index().to_dict()}')
    L.append('     within D km: ' + ' | '.join(f'{D} km {int((d["big_km"] <= D).sum())} ({100 * (d["big_km"] <= D).sum() / n:.0f}%)' for D in RV.D_KM)
             + f' | the same reach as the S3.1 river (class {RV.USE_CLASS}), at D={RV.USE_D}: {int(((d["big_km"] <= RV.USE_D) & (d["big_reach"] == d[f"r{RV.USE_CLASS}_reach"])).sum())}')
    L.append('')
c = T[(T['group'] == 'wh_cities') & (T['big_km'] <= RV.USE_D)].sort_values('big_km')
L.append(f'World Heritage Cities with a much larger river within {RV.USE_D} km ({len(c)}); own = largest class in the basin, big = class of that river, s31 = the S3.1 river:')
show = c[['label', 'own_min', 'big_ord', 'big_km', 'big_name', 'r5_name', 'r5_km']].copy()
show['same'] = (c['big_reach'] == c['r5_reach']).values
for col in ('big_km', 'r5_km'):
    show[col] = show[col].round(1)
L.append(show.head(40).to_string(index=False))
L.append('')
L.append('the ten places (class 5 | class 6 | much larger within 50 km):')
for n_, (la, lo) in C.PLACES.items():
    r = RIV.get(RV.key(la, lo))
    if r is None:
        L.append(f'   {n_}: not in the table'); continue
    f1 = lambda v: 'none' if pd.isna(v) else f'{v:.1f}'
    L.append(f"   {n_}: {f1(r['r5_km'])} km {r['r5_name'] if isinstance(r['r5_name'], str) else '-'} (line {f1(r['r5_name_km'])}) | {f1(r['r6_km'])} km {r['r6_name'] if isinstance(r['r6_name'], str) else '-'} | big class {'-' if pd.isna(r['big_ord']) else int(r['big_ord'])} {f1(r['big_km'])} km {r['big_name'] if isinstance(r['big_name'], str) else '-'} (own class {'-' if pd.isna(r['own_min']) else int(r['own_min'])})")
print('\n'.join(L))